# Controller v5 — W1–W5, T4 Run all

Bật **Secrets → GITHUB_TOKEN → Notebook access**, chọn **Tesla T4**, rồi **Runtime → Run all**. Chấp nhận quyền mount Drive. Notebook tự clone source private, pin full SHA, kiểm dữ liệu, build/test, chạy W1–W5, checkpoint và tải ZIP.

Giữ dữ liệu Drive cũ nếu còn. Nếu mất thư mục làm việc, đặt ZIP `full-1382a8b5cae5-68c68d47bc6c42ad93cc2974fcd59193.zip` vào `MyDrive/controller-score-results/`. ZIP diagnostics A–E có thể giữ trong `MyDrive/controller-diagnostics-AE/`. Ảnh cần cho v5 được khôi phục từ setup cũ/bundle; nếu thiếu, tự tải training archive đã dùng và kiểm lại hash/cohort. Không tự chạy lại full GPU sweep.

W1 là counterfactual offline; W2–W4 train/evaluate trên grouped development; W5 mới đo CUDA E2E. 50 pair evaluation cũ đã xem nên không được gọi là blind holdout. Default Controller được giữ nguyên, kể cả khi kết quả development tốt hơn.

In [ ]:
# CONFIG — only environment paths; no model/phase/policy choices.
CONFIG = {
    "REPO": "nh-atuan/crossstage-stitch",
    "REF": "de/optimize-controller",
    "DRIVE_ROOT": "/content/drive/MyDrive/controller-score-work",
    "OUTPUT_ROOT": "/content/drive/MyDrive/controller-v5-results",
    "ARTIFACT_PATH": "",
    "TRAINING_ZIP": "",
    "DIAGNOSTICS_PATH": "",
}

## Clone private và khóa source

Token Fine-grained cần Contents: Read-only cho repository. Token không nằm trong URL, remote, argv hoặc ZIP. Nếu không đọc được Secret/quyền repo, cell dừng kèm hướng dẫn.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import re
import shutil
import tempfile


def checkout_private(repository, ref, workspace, get_secret):
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository) or repository == "OWNER/REPO":
        raise RuntimeError("Điền CONFIG.REPO theo dạng OWNER/REPO.")
    if not ref or ref.startswith("-") or any(c in ref for c in "\r\n"):
        raise RuntimeError("Điền CONFIG.REF bằng branch hoặc commit hợp lệ đã push.")
    checkout = Path(workspace) / "controller-score-source"
    lock = checkout / ".git/score-checkout.json"
    existing = lock.exists()
    if existing:
        stored = json.loads(lock.read_text())
        actual = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(checkout), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
        if stored["repository"] != repository or stored["revision"] != actual or dirty:
            raise RuntimeError("Checkout đã đổi repo/HEAD hoặc có sửa tracked files. Giữ checkpoint trên Drive và dùng checkout sạch; không ghi đè thay đổi.")
        if re.fullmatch(r"[0-9a-f]{40}", ref) and actual == ref:
            return checkout, actual
    elif checkout.exists():
        raise RuntimeError("Checkout dở dang. Xóa riêng /content/controller-score-source trong Files rồi Run all; giữ checkpoint trên Drive.")
    try:
        token = get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Không đọc được GITHUB_TOKEN. Mở Secrets, thêm token và bật Notebook access; kiểm quyền repository.") from None
    if not isinstance(token, str) or not token.strip():
        token = None
        raise RuntimeError("GITHUB_TOKEN trống. Tạo Fine-grained PAT Contents: Read-only cho đúng repo trong Colab Secrets.")
    child_env = {k:v for k,v in os.environ.items() if not k.startswith("GIT_") and k not in ("GITHUB_TOKEN", "GH_TOKEN")}
    try:
        with tempfile.TemporaryDirectory(prefix="score-git-auth-") as auth:
            askpass = Path(auth) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['SCORE_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            child_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0", SCORE_GITHUB_TOKEN=token)
            args = ["git", "-c", "credential.helper=", "-c", "core.askPass="+str(askpass), "-c", "http.extraHeader="]
            commands = [] if existing else [args+["init", str(checkout)],
                args+["-C", str(checkout), "remote", "add", "origin", "https://github.com/"+repository+".git"]]
            commands += [args+["-C", str(checkout), "fetch", "--depth=1", "origin", ref],
                args+["-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]]
            for command in commands:
                completed = subprocess.run(command, env=child_env, capture_output=True, text=True)
                if completed.returncode:
                    # Suppress all network output; no credential-bearing output reaches a cell/log.
                    raise RuntimeError("Clone/checkout thất bại. Kiểm OWNER/REPO, ref đã push, Contents: Read-only và approval của tổ chức; không đổi sang token ghi hoặc URL chứa token.")
    finally:
        token = None
        child_env.pop("SCORE_GITHUB_TOKEN", None)
        child_env.pop("GIT_ASKPASS", None)
    revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Không khóa được full commit SHA")
    lock.write_text(json.dumps({"repository": repository, "ref": ref, "revision": revision}))
    return checkout, revision


from google.colab import userdata
SOURCE, REVISION = checkout_private(CONFIG["REPO"], CONFIG["REF"], "/content", userdata.get)
os.chdir(SOURCE)
if not (SOURCE/"tools/controller/v5/runner.py").is_file():
    raise RuntimeError("Ref chưa chứa Controller v5. Push commit đã bàn giao hoặc chọn branch/commit có tools/controller/v5/runner.py.")
print("LOCKED COMMIT:", REVISION)

import importlib
for _name in list(sys.modules):
    if _name == "tools" or _name.startswith("tools."):
        del sys.modules[_name]
importlib.invalidate_caches()
sys.path.insert(0, str(SOURCE))


## Mount Drive và dependency đã pin

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
    str(SOURCE / "tools/controller/v5/requirements.txt")], check=True)
# pip-installed CMake must be visible to the build subprocesses.
import shutil
if shutil.which("cmake") is None:
    raise RuntimeError("CMake chưa có trong PATH; xem dependency log.")


## Chạy toàn bộ W1 → W5

Seed 20261101, 2 warmup, 10 repeat, 7 E2E variants được khóa trong script. Kiểm T4, CUDA sm_75, CTest, pytest và C++/Python parity trước khi đo. W4 có nhiều fit CPU và log từng fold/seed/cỡ mẫu; không cần bấm riêng từng phần.

Checkpoint lưu trên Drive sau mỗi model/workstream/policy batch. Nếu runtime ngắt, mở lại notebook cùng CONFIG và Run all. Fingerprint khác code/data/model/GPU sẽ không ghép số đo cũ. Khi thiếu dữ liệu hoặc test fail, ZIP giữ trạng thái PARTIAL và lý do; không điền kết quả GPU giả.

In [ ]:
# Execute in a fresh interpreter after pip; no kernel restart is needed.
CONFIG_PATH = Path("/content/controller-v5-config.json")
RECEIPT_PATH = Path("/content/controller-v5-result.json")
STDOUT_LOG = Path("/content/controller-v5-notebook.log")
CONFIG_PATH.write_text(json.dumps({k: CONFIG[k] for k in [
    "REPO", "REF", "DRIVE_ROOT", "OUTPUT_ROOT", "ARTIFACT_PATH", "TRAINING_ZIP", "DIAGNOSTICS_PATH"]}, indent=2))
RECEIPT_PATH.unlink(missing_ok=True)
COMMAND = [sys.executable, "-u", "-m", "tools.controller.v5.runner",
    "--config", str(CONFIG_PATH), "--receipt", str(RECEIPT_PATH)]
with STDOUT_LOG.open("w") as log:
    process = subprocess.Popen(COMMAND, cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
if returncode or not RECEIPT_PATH.is_file():
    raise RuntimeError("Pipeline không khởi động/đóng gói được; xem " + str(STDOUT_LOG))
RESULT_ZIP = Path(json.loads(RECEIPT_PATH.read_text())["result_zip"])


## Tổng hợp, đóng gói và tải ZIP

Gửi nguyên ZIP lại để phân tích. ZIP gồm status W1–W5, model/fold/config hashes, CSV/JSON, PNG/PDF, attempts/profiles/QoR khi đã đo và log kiểm tra. Không gồm ảnh dataset hoặc token.

In [ ]:
from tools.controller.diagnostics.io import package, sha
RESULT_FOLDER = RESULT_ZIP.with_suffix("")
shutil.copy2(STDOUT_LOG, RESULT_FOLDER / "notebook-stdout.log")
RESULT_ZIP = package(RESULT_FOLDER)
print((RESULT_FOLDER / "workstream-status.csv").read_text())
print((RESULT_FOLDER / "README-results.md").read_text())
print("LOCKED SOURCE SHA:", REVISION)
print("DRIVE ZIP:", RESULT_ZIP)
print("ZIP SHA256:", sha(RESULT_ZIP))
from IPython.display import display, Image
for figure in ["W2/figures/fixed-config-auc.png", "W3/figures/gate-confusion.png",
    "W4/figures/pre_brier.png", "W5/figures/latency.png"]:
    path = RESULT_FOLDER / figure
    if path.is_file():
        display(Image(filename=str(path)))
# Keep the durable Drive copy; download a local copy when reasonably sized.
LOCAL_ZIP = Path("/content") / RESULT_ZIP.name
shutil.copy2(RESULT_ZIP, LOCAL_ZIP)
from google.colab import files
if LOCAL_ZIP.stat().st_size <= 250 * 1024 * 1024:
    files.download(str(LOCAL_ZIP))
else:
    print("ZIP lớn; tải bản Drive theo đường dẫn trên.")
